# EDA of results of SF Model v1

In [24]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sb
import numpy as np
import os
from itertools import product

In [26]:
df = pd.read_csv("./results_v2.csv")
# Cleaning
def clean_key(col):
    return col.str.replace(r'[\[\]",]', '', regex=True).str.strip()

split_keys = df['SERIES'].str.split('\n', expand=True)
df['key_1'] = clean_key(split_keys[1])
df['key_2'] = clean_key(split_keys[2])
df['key_3'] = clean_key(split_keys[3])

df['as_of_date'] = pd.to_datetime(df['TS']).dt.date
df['value'] = df['Y']
df = df.drop(columns=['SERIES', 'TS', 'Y'])

df.head()

,FORECAST,LOWER_BOUND,UPPER_BOUND,IS_ANOMALY,PERCENTILE,DISTANCE,key_1,key_2,key_3,as_of_date,value
0,1434.000169,1433.993847,1434.006491,False,0.014661,-2.179142,key_1_b,key_2_c,key_3_b,2026-01-17,1433.994820
1,1434.000352,1433.994030,1434.006674,False,0.778916,0.768538,key_1_b,key_2_c,key_3_b,2026-01-18,1434.002239
2,1434.000485,1433.994163,1434.006806,False,0.600459,0.254535,key_1_b,key_2_c,key_3_b,2026-01-19,1434.001109
3,1434.000684,1433.994362,1434.007006,False,0.088190,-1.351985,key_1_b,key_2_c,key_3_b,2026-01-20,1433.997366
4,1434.001555,1433.995233,1434.007877,False,0.176912,-0.927198,key_1_b,key_2_c,key_3_b,2026-01-21,1433.999279


In [27]:
allowed_key_1 = ['key_1_a', 'key_1_b', 'key_1_c']
allowed_key_2 = ['key_2_a', 'key_2_b', 'key_2_c']    
allowed_key_3 = ['key_3_a', 'key_3_b', 'key_3_c']    

dfs = []

for key1,key2,key3 in product(allowed_key_1, allowed_key_2, allowed_key_3):
    filtered_df = df[
        (df['key_1'] == key1) & 
        (df['key_2'] == key2) & 
        (df['key_3'] == key3)
    ]
    if not filtered_df.empty:
        dfs.append(filtered_df)
    
dfs    

[         FORECAST  LOWER_BOUND  UPPER_BOUND  IS_ANOMALY  PERCENTILE  DISTANCE  \
 7665  2824.000123  2823.994735  2824.005511       False    0.629861  0.331486   
 7666  2823.999872  2823.994485  2824.005260       False    0.058558 -1.566991   
 7667  2823.999770  2823.994382  2824.005158       False    0.863037  1.094064   
 7668  2823.999400  2823.994012  2824.004788       False    0.769949  0.738678   
 7669  2823.999982  2823.994594  2824.005370       False    0.600006  0.253361   
 ...           ...          ...          ...         ...         ...       ...   
 8025  2824.000554  2823.995166  2824.005942       False    0.666726  0.430890   
 8026  2824.000865  2823.995477  2824.006252       False    0.358046 -0.363687   
 8027  2824.000879  2823.995492  2824.006267       False    0.610324  0.280163   
 8028  2824.001166  2823.995778  2824.006554       False    0.262857 -0.634562   
 8029  2824.000854  2823.995466  2824.006242       False    0.369552 -0.333041   
 
         key_1

In [28]:
os.makedirs('./anomalies', exist_ok=True)
for idx, d in enumerate(dfs):
    if d.empty:
        continue
    
    
    d = d.sort_values('as_of_date')
    
    k1, k2, k3 = d['key_1'].iloc[0], d['key_2'].iloc[0], d['key_3'].iloc[0]
    
    plt.plot(d['as_of_date'], d['value'], label='Actual', color='#1f77b4', linewidth=1.5)
    plt.plot(d['as_of_date'], d['FORECAST'], label='Forecast', color='#ff7f0e', linestyle='--')
    plt.fill_between(
        d['as_of_date'], 
        d['LOWER_BOUND'], 
        d['UPPER_BOUND'], 
        color='gray', 
        alpha=0.2, 
        label='Prediction Interval'
    )
    
    anomalies = d[d['IS_ANOMALY'] == True]
    plt.scatter(
        anomalies['as_of_date'], 
        anomalies['value'], 
        color='red', 
        edgecolor='black', 
        s=50, 
        label='Anomaly', 
        zorder=5
    )
    
    plt.title(f"Anomaly Detection: {k1} | {k2} | {k3}")
    plt.xlabel("Date")
    plt.ylabel("Value")
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.xticks(rotation=45)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    
    plt.savefig(f'./anomalies/anomaly_plot_{idx}.png')
    plt.clf()

<Figure size 640x480 with 0 Axes>